# ViGoEmotions Midterm - Colab quick start (run everything)
Runs the complete Midterm pipeline on a Colab GPU: dataset analysis -> fixed split -> TF-IDF+LR ->
BiLSTM -> PhoBERT (+ optional ViSoBERT / CafeBERT / PhoBERT-large) -> ensemble -> evaluation & error analysis.

**Before running:** upload the project folder (including `data/train.csv`, `data/val.csv`, `data/test.csv`,
`data/label_dict.json`) to Google Drive and set `PROJECT_DIR` in the first cell.
Runtime -> Change runtime type -> **GPU** (T4 is enough for base models; L4/A100 for CafeBERT / PhoBERT-large).

> **Dataset licence.** ViGoEmotions is for non-commercial research/education only and must not be redistributed. Keep `data/` and `results/private/` out of any public repository; the tables in `results/tables/` contain only aggregates, ids and labels.

In [ ]:
# ---- Setup: works both locally and on Google Colab --------------------------------
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # EDIT: folder on your Drive that contains src/, configs/, resources/ and data/{train,val,test}.csv
    PROJECT_DIR = "/content/drive/MyDrive/vigoemotions_midterm"
    os.chdir(PROJECT_DIR)
    subprocess.run("pip -q install -r requirements-colab.txt", shell=True, check=True)
    # Java is needed by VnCoreNLP (PhoBERT word segmentation); falls back to pyvi otherwise
    if subprocess.run("java -version", shell=True, capture_output=True).returncode != 0:
        subprocess.run("apt-get -qq install -y default-jre-headless > /dev/null", shell=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
os.environ["PYTHONPATH"] = os.path.abspath("src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print("project dir:", os.getcwd())

import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
T = lambda name: pd.read_csv(f"results/tables/{name}")          # noqa: E731
F = lambda name, w=900: display(Image(f"results/figures/{name}", width=w))  # noqa: E731

In [ ]:
!nvidia-smi -L || echo 'no GPU'
!python -c "from vigo.utils import environment_info; import json; print(json.dumps(environment_info(), indent=1))"

## 1. Dataset analysis + fixed split

In [ ]:
!python -m vigo eda
!python -m vigo split --make iterative

## 2. Required models

In [ ]:
!python -m vigo train-tfidf --config configs/tfidf_lr.yaml

In [ ]:
!python -m vigo train-rnn --config configs/bilstm.yaml

In [ ]:
# PhoBERT-base-v2 (~6-8 min/epoch on T4 with fp16). Use train.amp=bf16 on L4/A100.
!python -m vigo train-transformer --config configs/phobert.yaml

## 3. Extra models for the best result (optional but recommended)

In [ ]:
!python -m vigo train-transformer --config configs/visobert.yaml
!python -m vigo train-transformer --config configs/visobert14gb.yaml

In [ ]:
# Large models: need a >= 16 GB GPU (L4 / A100). On T4 add model.gradient_checkpointing=true
# !python -m vigo train-transformer --config configs/cafebert.yaml
# !python -m vigo train-transformer --config configs/phobert_large.yaml

## 4. Ensemble (probability averaging, no test tuning)

In [ ]:
import os
members = [r for r in ['phobert', 'visobert', 'visobert14gb', 'cafebert', 'phobert_large']
           if os.path.exists(f'results/runs/{r}/summary.json')]
print(members)
ens = 'ensemble_3' if members == ['phobert', 'visobert', 'visobert14gb'] else 'ensemble_all'
!python -m vigo ensemble --runs {' '.join(members)} --name {ens}

## 5. Evaluation, Midterm questions, error analysis

In [ ]:
extra = ' '.join([r for r in ['visobert', 'visobert14gb', 'cafebert', 'phobert_large', 'ensemble_3', 'ensemble_all']
                   if os.path.exists(f'results/runs/{r}/summary.json')])
!python -m vigo analyze --tfidf tfidf_lr --rnn bilstm --transformer phobert --extra {extra}

In [ ]:
display(T("main_results_test.csv"))
display(T("threshold_strategies.csv"))